# 07_portfolio_backtesting

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from src.paths import PROCESSED_PATH
from src.risk import (
    annualized_return,
    annualized_volatility,
    sharpe_ratio,
    sortino_ratio,
    max_drawdown,
    calmar_ratio,
    recovery_time,
    drawdown_series
)
from src.backtest import (
    period_start_dates,
    simulate_dca,
    simulate_rebalanced_portfolio,
    xirr,
)

## 2. Paths & Config

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

from src.risk import RISK_FREE_RATE  # ECB deposit rate, single source in src/risk.py
TRADING_DAYS = 252
ROLLING_RETURN_WINDOW = 252 
REBALANCE_FREQUENCY = "YS"  # annual, see rationale in section 8
REBALANCE_LABEL = "Annual"
DCA_MONTHLY_AMOUNT = 100   
BENCHMARK_TICKER = "VWCE.DE"
BENCHMARK_LABEL = "Benchmark (FTSE All-World)"
TRANSACTION_COST_BPS = 10  

## 3. Strategy Metrics Engine

In [ ]:
def compute_strategy_metrics(
    returns,
    risk_free_rate=RISK_FREE_RATE,
    periods_per_year=TRADING_DAYS
):
    return {
        "Annualized Return": annualized_return(returns, periods_per_year),
        "Annualized Volatility": annualized_volatility(returns, periods_per_year),
        "Sharpe Ratio": sharpe_ratio(returns, risk_free_rate, periods_per_year),
        "Sortino Ratio": sortino_ratio(returns, risk_free_rate, periods_per_year),
        "Max Drawdown": max_drawdown(returns),
        "Calmar Ratio": calmar_ratio(returns, periods_per_year),
        "Recovery Time (Days)": recovery_time(returns)
    }

## 4. Load Strategy Returns (Current & Target from Notebook 03)

In [ ]:
risk_timeseries = pd.read_csv(DATA_PATH / "portfolio_risk_timeseries.csv", index_col=0, parse_dates=True)

strategy_returns = pd.DataFrame({
    "Current Weights (Static)": risk_timeseries["actual_portfolio_return"],
    "Target Weights (Static)": risk_timeseries["target_portfolio_return"]
})

asset_returns = pd.read_csv(DATA_PATH / "asset_returns_eur.csv", index_col=0, parse_dates=True)

strategy_returns.tail()

## 5. Benchmark Strategy

In [ ]:
benchmark_hist = yf.download(
    BENCHMARK_TICKER,
    start=strategy_returns.index.min(),
    end=strategy_returns.index.max() + pd.Timedelta(days=1),  # yfinance end is exclusive
    auto_adjust=True,
    progress=False
)

benchmark_close = benchmark_hist["Close"]

if isinstance(benchmark_close, pd.DataFrame):
    benchmark_close = benchmark_close.iloc[:, 0]

benchmark_returns = benchmark_close.pct_change().reindex(strategy_returns.index)

missing_days = int(benchmark_returns.isna().sum())
coverage = 1 - missing_days / len(benchmark_returns)
assert coverage > 0.95, f"benchmark covers only {coverage:.1%} of the backtest window"
print(f"{BENCHMARK_TICKER}: {missing_days} missing day(s), coverage {coverage:.1%}")

# Non-overlapping days (holiday calendar differences) are treated as flat.
strategy_returns[BENCHMARK_LABEL] = benchmark_returns.fillna(0.0)

## 6. Load & Validate Target Weights

In [ ]:
holdings = pd.read_csv(DATA_PATH / "current_holdings_trackable.csv")
target_weights_raw = pd.read_csv(DATA_PATH / "actual_vs_target_weights.csv")

target_weights = (
    target_weights_raw
    .merge(holdings[["symbol", "yf_ticker"]], on="symbol", how="left")
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")["target_weight"]
    .astype(float)
)
target_weights = target_weights.reindex(asset_returns.columns).fillna(0.0)

if target_weights.sum() <= 0:
    raise ValueError("Target weights must have a positive sum.")

print("Target weight sum before normalization:", target_weights.sum())
target_weights = target_weights / target_weights.sum()
assert abs(target_weights.sum() - 1.0) < 1e-10

## 7. Periodic Rebalancing

In [ ]:
rebalance_dates = period_start_dates(asset_returns.index, REBALANCE_FREQUENCY)
rebalanced_returns, turnover_series = simulate_rebalanced_portfolio(asset_returns, target_weights, rebalance_dates)
strategy_returns[f"Rebalanced Portfolio ({REBALANCE_LABEL})"] = rebalanced_returns
turnover_series

## 8. Compare Rebalancing Frequencies

In [ ]:
rebalance_frequency_results = []

for label, frequency in {
    "Monthly": "MS",
    "Quarterly": "QS",
    "Semiannual": "2QS",
    "Annual": "YS"
}.items():
    dates = period_start_dates(asset_returns.index, frequency)

    returns_frequency, turnover_frequency = simulate_rebalanced_portfolio(
        asset_returns, target_weights, dates
    )

    result = compute_strategy_metrics(returns_frequency)
    result["Strategy"] = label
    result["Average Turnover"] = turnover_frequency.mean()
    result["Total Turnover"] = turnover_frequency.sum()

    rebalance_frequency_results.append(result)

rebalance_frequency_summary = (
    pd.DataFrame(rebalance_frequency_results)
    .set_index("Strategy")
)

rebalance_frequency_summary

**Why annual rebalancing is the default.** Rebalancing less often lets winners run and keeps turnover low; rebalancing more often keeps the weights closer to target. The table above shows how the frequencies compare on your own portfolio. When reading it, keep two caveats in mind:

- The ratios are not independent evidence: they all come from the same return path, so a slightly higher return with unchanged volatility/drawdown lifts every one of them at once.
- Any edge is regime-dependent: letting winners run pays off in trending markets and reverses in mean-reverting ones, so a small gap between frequencies over one backtest window is weak evidence for the next one.

Annual is the default because it has the lowest turnover — and turnover is where the real costs sit that this notebook does not model: selling to rebalance triggers capital gains tax, on top of the transaction costs in section 9. Unless a higher frequency is clearly and consistently better, the cheaper option wins. In practice most rebalancing can happen buy-only through new contributions (Notebook 02), so a sell-side rebalance is only a once-a-year check.

## 9. Transaction Costs (Net Rebalancing)

In [ ]:
rebalance_costs = pd.Series(0.0, index=rebalanced_returns.index)
rebalance_costs.loc[turnover_series.index] = turnover_series * TRANSACTION_COST_BPS / 10_000
strategy_returns[f"Rebalanced Portfolio ({REBALANCE_LABEL}, Net)"] = rebalanced_returns - rebalance_costs
strategy_returns

## 10. Selection-Score Portfolio (In-Sample)

In [ ]:
# Uses today's selection scores applied across the whole history: look-ahead bias,
# so this is an in-sample upper bound, not an achievable strategy (see README, Key Concepts).
selection_ranking = pd.read_csv(DATA_PATH / "asset_selection_ranking.csv")

score_weights = (
    selection_ranking
    .dropna(subset=["selection_score", "yf_ticker"])
    .set_index("yf_ticker")["selection_score"]
    .astype(float)
    .clip(lower=0.0)  # long-only: don't treat negative scores as shorts
)
score_weights = score_weights.reindex(asset_returns.columns).fillna(0.0)

if score_weights.sum() <= 0:
    raise ValueError("No positive selection scores available.")

score_weights = score_weights / score_weights.sum()
assert abs(score_weights.sum() - 1.0) < 1e-10

strategy_returns["Selection-Score Portfolio (In-Sample)"] = asset_returns.mul(score_weights, axis=1).sum(axis=1)
strategy_returns

## 11. Cumulative Performance

In [ ]:
cumulative_growth = (1 + strategy_returns).cumprod()

plt.figure(figsize=(12, 5))

for strategy in cumulative_growth.columns:
    plt.plot(cumulative_growth.index, cumulative_growth[strategy], label=strategy)

plt.title("Backtest: Cumulative Portfolio Growth")
plt.xlabel("Date")
plt.ylabel("Growth of 1 €")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Drawdown Path

In [ ]:
strategy_drawdowns = strategy_returns.apply(drawdown_series)

plt.figure(figsize=(12, 5))

for strategy in strategy_drawdowns.columns:
    plt.plot(
        strategy_drawdowns.index,
        strategy_drawdowns[strategy] * 100,
        label=strategy
    )

plt.title("Strategy Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Rolling Returns

In [ ]:
rolling_returns = (
    (1 + strategy_returns)
    .rolling(ROLLING_RETURN_WINDOW)
    .apply(lambda x: x.prod() - 1, raw=True)
)

plt.figure(figsize=(12, 5))

for strategy in rolling_returns.columns:
    plt.plot(rolling_returns.index, rolling_returns[strategy] * 100, label=strategy)

plt.title(f"Rolling {ROLLING_RETURN_WINDOW}-Day Return")
plt.xlabel("Date")
plt.ylabel("Rolling Return (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 14. Overall Metrics

In [ ]:
metrics_summary = pd.DataFrame({
    strategy: compute_strategy_metrics(strategy_returns[strategy])
    for strategy in strategy_returns.columns
}).T

overall_metrics = metrics_summary[["Annualized Return", "Annualized Volatility", "Sharpe Ratio", "Sortino Ratio", "Calmar Ratio", "Max Drawdown"]]
overall_metrics

## 15. DCA Simulation

In [ ]:
# include_first: the first (partial) month gets its contribution too
dca_dates = period_start_dates(asset_returns.index, "MS", include_first=True)

dca_result = simulate_dca(asset_returns, target_weights, dca_dates, DCA_MONTHLY_AMOUNT)

dca_result["gain"] = dca_result["portfolio_value"] - dca_result["total_invested"]
dca_result["simple_gain_on_contributions"] = (
    dca_result["gain"] / dca_result["total_invested"].replace(0, pd.NA)
)

dca_result.tail()

In [ ]:
contribution_dates = list(dca_dates)
contribution_cashflows = [-DCA_MONTHLY_AMOUNT] * len(contribution_dates)

dca_cashflow_dates = contribution_dates + [dca_result.index[-1]]
dca_cashflows = contribution_cashflows + [dca_result["portfolio_value"].iloc[-1]]

dca_xirr = xirr(dca_cashflow_dates, dca_cashflows)
print(f"DCA XIRR: {dca_xirr:.2%}")

dca_backtest_summary = pd.Series({
    "start_date": dca_result.index.min().date().isoformat(),
    "monthly_amount": DCA_MONTHLY_AMOUNT,
    "total_invested": dca_result["total_invested"].iloc[-1],
    "final_value": dca_result["portfolio_value"].iloc[-1],
    "gain": dca_result["gain"].iloc[-1],
    "xirr": dca_xirr,
}, name="value")

dca_backtest_summary

## 16. Sanity Check vs. Notebook 03

In [ ]:
existing_metrics = pd.read_csv(
    DATA_PATH / "portfolio_risk_performance_metrics.csv",
    index_col=0
)

sanity_check_map = {
    "Current Weights (Static)": "Current Portfolio",
    "Target Weights (Static)": "Target Portfolio"
}

comparison = pd.DataFrame({
    new_name: (
        metrics_summary.loc[new_name, existing_metrics.columns].astype(float)
        - existing_metrics.loc[old_name].astype(float)
    )
    for new_name, old_name in sanity_check_map.items()
}).T.round(6)

assert (comparison.abs() < 1e-6).all().all(), (
    "Mismatch vs. Notebook 03 metrics - check the calculation."
)

print("Matches Notebook 03.")
comparison

## 17. Export

In [ ]:
strategy_returns.to_csv(DATA_PATH / "backtest_strategy_returns.csv")
metrics_summary.to_csv(DATA_PATH / "backtest_metrics_summary.csv")
turnover_series.to_csv(DATA_PATH / "backtest_rebalance_turnover.csv")
dca_result.to_csv(DATA_PATH / "backtest_dca_simulation.csv")
cumulative_growth.to_csv(DATA_PATH / "backtest_cumulative_growth.csv")
rolling_returns.to_csv(DATA_PATH / "backtest_rolling_returns.csv")
strategy_drawdowns.to_csv(DATA_PATH / "backtest_drawdowns.csv")
rebalance_frequency_summary.to_csv(DATA_PATH / "backtest_rebalance_frequency_summary.csv")
dca_backtest_summary.to_csv(DATA_PATH / "backtest_dca_summary.csv")

print("All backtest exports saved.")